## Popcorn Hack 1: Fix the Beaker Row
Replaced the inline flex style and made-up `box` class with one standard grid and three grid cells.

In [ ]:
%%html
<!-- UI_RUNNER: Grids Popcorn 1 -->

<div class="ocs__grid ocs__grid--standard">
  <div class="ocs__grid-cell">Beaker A: Water</div>
  <div class="ocs__grid-cell">Beaker B: Salt Water</div>
  <div class="ocs__grid-cell">Beaker C: Vinegar</div>
</div>

## Popcorn Hack 2: Size the Beakers with `fr`
4 `fr` values = 4 boxes in one row. The Control Beaker gets `2fr`, so it is twice as wide as each `1fr` beaker. (The lesson allows an inline style only for this `fr` hack.)

In [ ]:
%%html
<!-- UI_RUNNER: Grids Popcorn 2 -->

<div class="ocs__grid ocs__grid--standard" style="grid-template-columns: 1fr 1fr 1fr 2fr;">
  <div class="ocs__grid-cell">Beaker 1</div>
  <div class="ocs__grid-cell">Beaker 2</div>
  <div class="ocs__grid-cell">Beaker 3</div>
  <div class="ocs__grid-cell">Control Beaker</div>
</div>

## MCQ Check

MCQ Grids: 4/4 | answers: A,B,C,D

| # | Answer | Why |
|---|---|---|
| 1 | A `cols-4` | Changes a standard grid to 4 columns (same as `1fr 1fr 1fr 1fr`) |
| 2 | B 2 boxes, first twice as wide | Two values = two boxes; 2 of 3 shares go to the first |
| 3 | C `--header` | Header spans every column; `--wide` only spans 2 |
| 4 | D `my-row` + `display:flex` | Made-up class and inline style break the rule |

## Homework Hack
Both grids sit inside one `ocs__card`. Grid 1 is a standard grid: a header title, the outlier Beaker 3 as accent, the control reading as muted, and a notes cell that is wide (spans 2 columns). Grid 2 is a calculator keypad with an accent `Avg` key. No inline styles, no made-up classes, `ocs__grid-cell` on every box.

In [ ]:
%%html
<!-- UI_RUNNER: Grids Homework -->

<div class="ocs__card">
  <!-- Grid 1: standard grid (3 columns) of beaker readings -->
  <div class="ocs__grid ocs__grid--standard">
    <div class="ocs__grid-cell ocs__grid-cell--header">Temperature Trial (°C)</div>
    <div class="ocs__grid-cell">Beaker 1: 22.4</div>
    <div class="ocs__grid-cell">Beaker 2: 22.9</div>
    <div class="ocs__grid-cell ocs__grid-cell--accent">Beaker 3: 31.6</div>
    <div class="ocs__grid-cell ocs__grid-cell--muted">Control: 22.5</div>
    <div class="ocs__grid-cell ocs__grid-cell--wide">Note: Beaker 3 is about 9 °C above the control, so it is the outlier.</div>
  </div>

  <!-- Grid 2: calculator keypad (4 columns) -->
  <div class="ocs__grid ocs__grid--calculator">
    <div class="ocs__grid-cell">7</div>
    <div class="ocs__grid-cell">8</div>
    <div class="ocs__grid-cell">9</div>
    <div class="ocs__grid-cell ocs__grid-cell--accent">Avg</div>
  </div>
</div>

| Original | Refactored |
|---|---|
| Loose grids on the page | One `ocs__card` holding both grids |
| `my-row`, `display: grid` inline | `ocs__grid ocs__grid--standard` / `ocs__grid--calculator` |
| `title-box` with `width: 100%` | `ocs__grid-cell--header` (spans the whole row) |
| `sample-card highlight` | `ocs__grid-cell ocs__grid-cell--accent` (outlier) |
| Control reading styled by hand | `ocs__grid-cell--muted` (reference value, quieter) |
| Notes box with a set width | `ocs__grid-cell--wide` (spans 2 columns) |
| `key`, `key orange` | `ocs__grid-cell`, `ocs__grid-cell--accent` (Avg) |

## The SASS behind my grids

The page only uses OCS grid classes (no inline styles beyond the lesson's `fr` popcorn). This is the SASS that powers them, based on OCS `elements/grids/_main.scss`: a **`standard-grid($cols)` mixin** builds an N-column grid from `fr` units, an **`@each` loop** generates `cols-2`, `cols-3`, and `cols-4`, a **boolean + `@if`** collapses grids to one column on phones, and the cell modifiers (`--header`, `--accent`, `--muted`) are what I used for the title, the outlier beaker, and the `Avg` key.

```scss
// ---- Settings ----
$grid-gap: 1rem;
$breakpoint-mobile: 600px;
$grid-column-counts: (2, 3, 4);   // the cols-N options OCS offers
$grid-stack-on-mobile: true;      // boolean: collapse to 1 column on phones

// ---- Mixin: build an N-column grid from one rule ----
@mixin standard-grid($cols) {
  display: grid;
  grid-template-columns: repeat($cols, minmax(0, 1fr)); // N equal fr shares
  gap: $grid-gap;

  @if $grid-stack-on-mobile {
    @media (max-width: $breakpoint-mobile) {
      grid-template-columns: 1fr;
    }
  }
}

// Standard grid: 3 columns by default (used for my beaker readings)
.ocs__grid--standard {
  @include standard-grid(3);

  // @each loop: generates .cols-2, .cols-3, .cols-4 without repeating code
  @each $n in $grid-column-counts {
    &.cols-#{$n} {
      @include standard-grid($n);
    }
  }
}

// Calculator variant: always a 4-key-wide keypad (used for my calculator)
.ocs__grid--calculator {
  display: grid;
  grid-template-columns: repeat(4, 1fr);
  gap: 0.5rem;
}

// Cells and cell types
.ocs__grid-cell {
  padding: 1rem;
  border: 1px solid var(--ui-border);
  border-radius: 0.5rem;
  background: var(--panel);

  &--header { grid-column: 1 / -1; font-weight: 600; } // spans every column (my title)
  &--wide   { grid-column: span 2; }                    // takes 2 columns
  &--accent { background: color-mix(in srgb, var(--pref-accent-color) 18%, var(--panel)); } // my outlier / Avg key
  &--muted  { background: color-mix(in srgb, var(--pref-text-color) 6%, var(--panel)); }
}
```